# Energy Storage Intraday Trading: Global Optimization

In this notebook, we break down the **Linear Programming (LP)** approach to the energy storage intraday trading challenge, step by step.

By modeling our battery's physical constraints and trading goals mathematically, we can feed the entire year of 15-minute price intervals into a global optimization solver. This ensures we extract the absolute maximum profit, while strictly adhering to both local capacity limits and global cycle goals (without trapping ourselves in local optima like the greedy combinations approach did!).

### 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog
from scipy.sparse import lil_matrix
import matplotlib.pyplot as plt
import seaborn as sns

### 2. Load the Market Data
Let's load the 15-minute interval energy pricing data for Germany (`market_prices.csv`).

In [ ]:
df = pd.read_csv('market_prices.csv', sep=';', parse_dates=['timestamp_UTC'], dayfirst=True)
df = df.rename(columns={df.columns[1]: 'EUR/MWh'})
price_data = df['EUR/MWh'].values
n = len(price_data)
num_days = n // 96

print(f"Loaded {n} price intervals across {num_days} days.")
df.head()

### 3. Define the Operational Parameters
We declare the strict rules the energy storage asset must follow.

In [ ]:
nominal = 2  # Nominal charging/discharging power (MW)
capacity = 4  # Total usable capacity (MWh)
efficiency = 0.9  # 90% one-way efficiency
dt = 0.25  # The duration of each timestep in hours (15 minutes)

max_cycles_per_day = 2.5  # We cannot exceed 2.5 cycles on any single day
target_average_cycles = 1.5  # We must average exactly 1.5 cycles mathematically across the entire year

### 4. Build the Objective Function
For our LP solver, we need to declare the variables and the objective cost. We have three decision variables per time step $t$:
1. **$P_{ch}(t)$**: Power charged
2. **$P_{dis}(t)$**: Power discharged
3. **$E(t)$**: The State of Charge (Energy level)

Since the solver expects a massive 1D array of variables, our variables will sequence as:
`[P_ch(0), P_dis(0), E(0), P_ch(1), P_dis(1), E(1), ...]`

We want to **Maximize Profit**, but the software natively **Minimizes**. So we multiply the profit formula by $-1$.

In [ ]:
# Initialize a massive cost array of zeroes (size 3 * 35040 variables)
c = np.zeros(3 * n)

for t in range(n):
    # Note: Cost to charge is positive (spending money), Revenue to discharge is negative (earning money)
    c[3 * t] = price_data[t] * dt       # Cost applied to P_ch(t)
    c[3 * t + 1] = -price_data[t] * dt  # Revenue applied to P_dis(t)

### 5. Establish Fixed Bounds
Every variable has strict minimums and maximums (e.g., $P_{ch}$ and $P_{dis}$ cannot exceed the 2MW nominal limit, and $E$ cannot fall below 0 or exceed 4MWh).

In [ ]:
bounds = []
for t in range(n):
    bounds.append((0, nominal))   # Bounds for P_ch(t)
    bounds.append((0, nominal))   # Bounds for P_dis(t)
    bounds.append((0, capacity))  # Bounds for E(t)

### 6. Construct the Equality Constraints (Energy Balance & Targets)
The battery is a physical asset with intertemporal links. What you discharge in step $t$ directly changes the available state of charge in step $t+1$. We must also establish our strict long-term cycle target.

Because evaluating 105,000 continuous variables requires massive memory, we **must use a Sparse matrix representation** (`lil_matrix`) so the RAM allocation doesn't crash our system!

In [ ]:
# The matrix has 'n' rows for the physical battery state at each step, plus 1 row for the global average requirement.
A_eq_sparse = lil_matrix((n + 1, 3 * n))
b_eq = np.zeros(n + 1)

# 1. Sequential physical energy balance: 
# E(t) - E(t-1) - (P_ch(t) * eff * dt) + (P_dis(t) / eff * dt) = 0
for t in range(n):
    A_eq_sparse[t, 3 * t + 2] = 1.0
    A_eq_sparse[t, 3 * t] = -efficiency * dt
    A_eq_sparse[t, 3 * t + 1] = (1.0 / efficiency) * dt
    if t > 0:
        A_eq_sparse[t, 3 * (t - 1) + 2] = -1.0
        
# 2. Global Average Cycles Requirement:
# The total energy discharged over the whole year must equal exactly 1.5 cycles * capacity * 365 days.
for t in range(n):
    A_eq_sparse[n, 3 * t + 1] = dt
b_eq[n] = target_average_cycles * capacity * num_days

### 7. Construct the Inequality Constraints (Daily Cycle Caps)
Finally, while the total year averages 1.5 cycles a day, no *individual* day algorithmically chosen can ever exceed 2.5 cycles. We structure these caps as limits.

In [ ]:
# We have 365 inequality rules (one for each specific day)
A_ub_sparse = lil_matrix((num_days, 3 * n))
b_ub = np.full(num_days, max_cycles_per_day * capacity)

for d in range(num_days):
    # For every 15 min interval belonging to the current day 'd'
    for t in range(d * 96, (d + 1) * 96):
        if t < n:
            A_ub_sparse[d, 3 * t + 1] = dt  # P_dis(t) * dt

### 8. Run the Global Optimization!
With all parameters configured as large linear arrays, we can boot up SciPy's robust `Highs` algorithm. This resolves the mathematically perfect solution taking all variables and dependencies down sequentially to find the absolute optimum global profit.

In [ ]:
print("Starting Highs Linear Programming Solver...")
res = linprog(c, A_ub=A_ub_sparse, b_ub=b_ub, A_eq=A_eq_sparse, b_eq=b_eq, bounds=bounds, method='highs')

if res.success:
    cost = res.fun
    x = res.x
    total_p_ch = sum(x[3*t] * dt for t in range(n))
    total_p_dis = sum(x[3*t+1] * dt for t in range(n))
    total_purchase_cost = sum(x[3*t] * dt * price_data[t] for t in range(n))
    total_sales_revenue = sum(x[3*t+1] * dt * price_data[t] for t in range(n))
    cycle_cost = (total_purchase_cost - total_sales_revenue) / total_p_ch
    
    print(f"\n🎉 Global Optimization Successful!")
    print(f"--------------------------------------")
    print(f"Optimal Profit Over {num_days} Days: €{abs(cost):,.2f}")
    print(f"Cycle Cost (Net / MWh Charged): €{cycle_cost:,.2f}")
    print(f"Total Energy Discharged: {total_p_dis:,.2f} MWh")
    print(f"Total Full Equivalent Cycles: {total_p_dis / capacity:,.2f}")
    print(f"Average Daily Cycles Confirmed: {(total_p_dis / capacity) / num_days:,.4f}")
    
else:
    print(f"❌ Solver explicitly failed with message: {res.message}")